In [12]:
import pandas as pd
import os
import numpy as np
import shutil

# --- Configuration ---
# Set the input and output folder paths.
# Assuming your CSV files are in the same directory as this script.
input_folder = os.getcwd()
output_folder = "output"

# Columns to be dropped from the raw data
columns_to_drop = ['Series', 'No. of Trades', 'Turnover ₹', 'Deliverable Qty']

# --- Main Script ---
# Delete the output folder if it exists, to ensure a clean run.
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)
    print(f"Deleted existing output folder: {output_folder}")

# Create the output folder
os.makedirs(output_folder)
print(f"Created new output folder: {output_folder}")

# Get a list of all CSV files in the input folder
csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

if not csv_files:
    print("No CSV files found in the current directory. Please make sure your files are there.")
else:
    # A list to hold the processed DataFrames
    all_dfs = []
    
    for file_name in csv_files:
        print(f"Processing file: {file_name}")
        try:
            # Read the CSV file into a pandas DataFrame
            file_path = os.path.join(input_folder, file_name)
            df = pd.read_csv(file_path)

            # Convert 'Date' column to datetime objects.
            df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
            df = df.sort_values(by='Date')
            
            # Remove any rows with invalid dates after coercion
            df.dropna(subset=['Date'], inplace=True)

            # Drop the initial unnecessary columns
            df = df.drop(columns=columns_to_drop, errors='ignore')

            # --- Intermediate Calculations for Normalization ---
            # These are temporary and will be dropped later
            df['20_SMA'] = df['Close Price'].rolling(window=20).mean()
            df['20_STD'] = df['Close Price'].rolling(window=20).std()

            # --- Price-Agnostic Feature Engineering ---
            # ... (all feature engineering code from before is here) ...
            
            # --- Price Movement Features (Normalized) ---
            # Weekly Return as a percentage change
            df['Weekly_Return'] = df['Close Price'].pct_change() * 100
            
            # Normalize Open, High, and Low prices against the Close Price
            df['Normalized_Open'] = df['Open Price'] / df['Close Price']
            df['Normalized_High'] = df['High Price'] / df['Close Price']
            df['Normalized_Low'] = df['Low Price'] / df['Close Price']
            
            # Normalize EMAs as a percentage difference from the Close Price. Periods are now in weeks.
            ema5 = df['Close Price'].ewm(span=5, adjust=False).mean()
            ema13 = df['Close Price'].ewm(span=13, adjust=False).mean()
            ema23 = df['Close Price'].ewm(span=23, adjust=False).mean()
            df['EMA5_Diff_Pct'] = (df['Close Price'] - ema5) / df['Close Price'] * 100
            df['EMA13_Diff_Pct'] = (df['Close Price'] - ema13) / df['Close Price'] * 100
            df['EMA23_Diff_Pct'] = (df['Close Price'] - ema23) / df['Close Price'] * 100

            # --- Volume-Based Features ---
            # Volume SMA over 20 weeks
            df['Volume_SMA20'] = df['Total Traded Quantity'].rolling(window=20).mean()
            df['Relative_Volume'] = df['Total Traded Quantity'] / df['Volume_SMA20']
            
            # Volume Spike Ratio (5-week average)
            df['Volume_SMA5'] = df['Total Traded Quantity'].rolling(window=5).mean()
            df['Volume_Spike_Ratio_5Wk'] = df['Total Traded Quantity'] / df['Volume_SMA5']
            df = df.drop(columns=['Volume_SMA5'], errors='ignore')

            # On-Balance Volume (OBV)
            # A running total of volume based on price direction
            obv_series = pd.Series(0, index=df.index)
            obv_series.iloc[0] = df['Total Traded Quantity'].iloc[0]
            for i in range(1, len(df)):
                if df['Close Price'].iloc[i] > df['Close Price'].iloc[i-1]:
                    obv_series.iloc[i] = obv_series.iloc[i-1] + df['Total Traded Quantity'].iloc[i]
                elif df['Close Price'].iloc[i] < df['Close Price'].iloc[i-1]:
                    obv_series.iloc[i] = obv_series.iloc[i-1] - df['Total Traded Quantity'].iloc[i]
                else:
                    obv_series.iloc[i] = obv_series.iloc[i-1]
            df['OBV'] = obv_series
            # To make OBV price-agnostic, we'll normalize it by its 20-week SMA
            df['Normalized_OBV'] = df['OBV'] / df['OBV'].rolling(window=20).mean()
            
            # --- Volatility-Based Features ---
            # Normalized Weekly Range (Range as a percentage of the Close Price)
            df['Normalized_Weekly_Range'] = (df['High Price'] - df['Low Price']) / df['Close Price'] * 100
            
            # Average True Range (ATR) over a 14-week period
            df['High-Low'] = df['High Price'] - df['Low Price']
            df['High-PrevClose'] = abs(df['High Price'] - df['Close Price'].shift(1))
            df['Low-PrevClose'] = abs(df['Low Price'] - df['Close Price'].shift(1))
            df['True_Range'] = df[['High-Low', 'High-PrevClose', 'Low-PrevClose']].max(axis=1)
            df['ATR'] = df['True_Range'].ewm(span=14, adjust=False).mean()
            df = df.drop(columns=['High-Low', 'High-PrevClose', 'Low-PrevClose', 'True_Range'], errors='ignore')
            
            # Bollinger Band Z-Score based on 20-week SMA
            df['Bollinger_Z_Score'] = (df['Close Price'] - df['20_SMA']) / df['20_STD']
            
            # --- Momentum-Based Features ---
            # Rate of Change (ROC) over 10 weeks
            df['ROC_10'] = df['Close Price'].pct_change(periods=10) * 100
            
            # Relative Strength Index (RSI) over a 14-week period
            delta = df['Close Price'].diff(1)
            gain = delta.where(delta > 0, 0)
            loss = -delta.where(delta < 0, 0)
            avg_gain = gain.ewm(span=14, adjust=False).mean()
            avg_loss = loss.ewm(span=14, adjust=False).mean()
            rs = avg_gain / avg_loss
            df['RSI'] = 100 - (100 / (1 + rs))

            # Stochastic Oscillator (%K and %D)
            lowest_low = df['Low Price'].rolling(window=14).min()
            highest_high = df['High Price'].rolling(window=14).max()
            df['Stochastic_%K'] = ((df['Close Price'] - lowest_low) / (highest_high - lowest_low)) * 100
            df['Stochastic_%D'] = df['Stochastic_%K'].rolling(window=3).mean()
            
            # VWAP Difference as a percentage
            df['VWAP'] = (df['Close Price'] * df['Total Traded Quantity']).cumsum() / df['Total Traded Quantity'].cumsum()
            df['VWAP_Diff_Pct'] = (df['Close Price'] - df['VWAP']) / df['VWAP'] * 100

            # --- Final Cleanup: Drop all raw price columns and temporary calculations ---
            df = df.drop(columns=['Open Price', 'High Price', 'Low Price', 'Close Price', 
                                  'Last Price', 'Average Price', '20_SMA', '20_STD', 'VWAP', 'OBV'], errors='ignore')

            # Append the processed DataFrame to the list
            all_dfs.append(df)
            
            print(f"Successfully processed {file_name} and added to the list.\n")

        except Exception as e:
            print(f"An error occurred while processing {file_name}: {e}\n")

    if all_dfs:
        # Combine all DataFrames into a single one
        combined_df = pd.concat(all_dfs, ignore_index=True)
        
        # Sort the entire DataFrame by the Date column
        combined_df = combined_df.sort_values(by=['Date', 'Symbol']).reset_index(drop=True)
        
        # Save the final combined file
        output_file_path = os.path.join(output_folder, "combined_processed_data.csv")
        combined_df.to_csv(output_file_path, index=False)
        
        print(f"All processed data combined and saved to: {output_file_path}")

print("All files have been processed and combined.")


Deleted existing output folder: output
Created new output folder: output
Processing file: Tata-Combined-Weekly-Data.csv
Successfully processed Tata-Combined-Weekly-Data.csv and added to the list.

Processing file: Bajaj-Combined-Weekly-Data.csv
Successfully processed Bajaj-Combined-Weekly-Data.csv and added to the list.

Processing file: Hero-Combined-Weekly-Data.csv
Successfully processed Hero-Combined-Weekly-Data.csv and added to the list.

Processing file: Maruti-Combined-Weekly-Data.csv
Successfully processed Maruti-Combined-Weekly-Data.csv and added to the list.

Processing file: MM-Combined-Weekly-Data.csv
Successfully processed MM-Combined-Weekly-Data.csv and added to the list.

Processing file: Eicher-Combined-Weekly-Data.csv
Successfully processed Eicher-Combined-Weekly-Data.csv and added to the list.

All processed data combined and saved to: output/combined_processed_data.csv
All files have been processed and combined.
